# Entrenamiento y evaluación de un NER propio con spaCy: órdenes de trabajo

**Curso:** Procesamiento de Lenguaje Natural · Unidad 4 — Etiquetado de secuencias y reconocimiento de entidades nombradas  
**Institución:** Universidad Tecnológica de Bolívar  
**Referencia:** Jurafsky, D. & Martin, J. H. (2025). *Speech and Language Processing* (3.ª ed.), Cap. 17 — Sequence Labeling for Parts of Speech and Named Entities  
**Compatibilidad con Python:** 3.10 · 3.11 · 3.12 · 3.13

## Contexto y Propósito

Este notebook muestra cómo **entrenar un reconocedor de entidades nombradas (NER) propio con spaCy** para categorías que los modelos genéricos no conocen. El caso son órdenes de trabajo de una empresa de energía ficticia, donde se quieren extraer el **equipo** (transformador, reconectador…), la **ubicación** (subestación, circuito, barrio) y la **falla** (sobrecalentamiento, fuga de aceite…). Se verifican las anotaciones por posiciones de caracteres, se entrena un *pipeline* en blanco con la API `Example`, y se **evalúa** con oraciones redactadas con plantillas que el modelo no vio durante el entrenamiento.

**Objetivo:** aprender el procedimiento completo para crear un NER con categorías propias de un negocio, detectar errores de anotación antes de entrenar y medir honestamente si el modelo generaliza a redacciones nuevas.

## Ejemplos de Aplicación en el Mundo Real

Cada caso muestra, paso a paso, cómo se aplica la técnica de este notebook a un problema real.

### Energía e industria — Equipos en órdenes de trabajo

- **Problema:** el modelo general de spaCy no reconoce equipos eléctricos.
- **Entrada:** órdenes de trabajo (`cambio de reconectador en circuito 12`).
- **Cómo se aplica el entrenamiento de NER en spaCy:**
  1. Se anotan ejemplos con posiciones de caracteres (inicio, fin, etiqueta) para `EQUIPO`, `UBICACION` y `FALLA`.
  2. Se verifica que cada anotación coincida con límites de tokens antes de entrenar.
  3. Se entrena con la API `Example`, se guarda el modelo y se prueba con órdenes nuevas.
- **Salida:** equipo, ubicación y falla de cada orden.
- **Decisión:** estadísticas de fallas por tipo de equipo.

### Retail, agro y puertos — Productos en pedidos por chat

- **Problema:** una ferretería recibe pedidos por WhatsApp en lenguaje libre.
- **Entrada:** `mándame 3 bultos de cemento Argos`.
- **Cómo se aplica el entrenamiento de NER en spaCy:**
  1. Se anotan PRODUCTO, CANTIDAD y MARCA por caracteres.
  2. Se verifica la alineación de las anotaciones con los tokens.
  3. Se entrena el NER propio y se evalúa con mensajes no vistos.
- **Salida:** líneas del pedido (producto, cantidad, marca).
- **Decisión:** pedido prellenado que el vendedor confirma.

## Requisitos Previos

**Instalación** (una sola vez, desde la terminal; en Jupyter o Colab antepón `!`):

```bash
pip install spacy pandas
```

**Librerías que se importan:**

| Librería | Para qué se usa |
|---------|-----------------|
| `json, random, collections.Counter, pathlib.Path` | Lectura de anotaciones, barajado, conteos y rutas |
| `spacy` | Pipeline en blanco, componente NER, entrenamiento, guardado y carga |
| `spacy.training.Example, offsets_to_biluo_tags` | Ejemplos de entrenamiento y verificación de la alineación |
| `pandas` | Tablas de datos y de resultados |

**Datos:** `data/ordenes_trabajo_ner.json` (incluido en el repositorio): 80 órdenes de trabajo **ficticias**, generadas con 10 plantillas, con las etiquetas `EQUIPO`, `UBICACION` y `FALLA`.

## Marco metodológico: CRISP-ML(Q)

Este notebook sigue **CRISP-ML(Q)** (*Cross-Industry Standard Process for the development of Machine Learning applications with Quality assurance*; Studer et al., 2021). El proceso tiene seis fases, y en cada una se identifican los riesgos de calidad y cómo se mitigan. Las fases aparecen marcadas con 🔷 a lo largo del notebook.

| Fase CRISP-ML(Q) | Dónde se aborda en este notebook | Aseguramiento de calidad |
|------------------|----------------------------------|--------------------------|
| 1. Entendimiento del negocio y de los datos | Secciones 2 y 3: formato de las anotaciones, carga, distribución de etiquetas y **verificación de la alineación** | Toda entidad desalineada se reporta antes de entrenar |
| 2. Ingeniería de datos | Sección 4: partición por plantillas y registro de etiquetas | Las plantillas de prueba no aparecen en entrenamiento: se evita una evaluación optimista |
| 3. Ingeniería del modelo | Sección 5: pipeline en blanco y bucle de entrenamiento | Semilla fija y datos barajados en cada época |
| 4. Aseguramiento de calidad del modelo | Sección 6: precisión, *recall* y F1 por etiqueta en prueba, comparados con entrenamiento, y análisis de errores | La diferencia entre entrenamiento y prueba mide cuánto memoriza el modelo |
| 5. Despliegue | Sección 7: guardado con `to_disk` y carga con `spacy.load` | Se verifica que el modelo recargado dé los mismos resultados |
| 6. Monitoreo y mantenimiento | Al final (fuera de alcance) | — |

## Estándares de referencia

Prácticas de este notebook alineadas con estándares de IEEE SA, ISO/IEC y la Open Source Initiative (OSI). Se citan como guía de buenas prácticas, no como certificación de cumplimiento.

| Estándar | Cómo se aplica aquí |
|----------|---------------------|
| **ISO/IEC 23894:2023** — Guía para la gestión de riesgos de IA | Riesgos tratados: anotaciones desalineadas (sección 3.1) y evaluación optimista por plantillas repetidas (sección 4). |
| **IEEE 7001-2021** — Transparencia de sistemas autónomos | Se explica cómo se representa cada entidad (BILUO) y se muestran los errores del modelo en prueba. |
| **ISO/IEC 42001:2023** — Sistema de gestión de IA en una organización | Se documenta que los datos son sintéticos y que el resultado no garantiza el desempeño con órdenes reales. |
| **OSI · Open Source AI Definition 1.0 (2024)** — Qué hace que un sistema de IA sea de código abierto: libertades de usar, estudiar, modificar y compartir, con acceso a código, parámetros e información de los datos | spaCy es de código abierto (MIT) y el modelo entrenado se puede compartir junto con sus datos de entrenamiento. |

## Objetivos de aprendizaje

Al finalizar este notebook podrás:

1. Dar formato a datos anotados para entrenar NER en spaCy (entidades como posiciones de caracteres)
2. **Verificar** que cada anotación esté alineada con los límites de los tokens
3. **Diseñar** una partición de prueba que evite una evaluación optimista (plantillas no vistas)
4. Crear un *pipeline* vacío de spaCy con un componente **NER** y entrenarlo con la API `Example`
5. **Evaluar** el modelo con precisión, *recall* y F1 por etiqueta en entrenamiento y en prueba, e **interpretar** la diferencia
6. Guardar y recargar el modelo entrenado, y hacer inferencia

## ¿Qué es el reconocimiento de entidades nombradas (NER)?

NER es una tarea de etiquetado de secuencias que identifica y clasifica **entidades** en un texto. Los modelos genéricos reconocen personas, lugares y organizaciones; un NER propio aprende las categorías de un negocio, como equipos o fallas. El NER de spaCy es un reconocedor **basado en transiciones** sobre una red neuronal.

> ℹ️ Las órdenes de trabajo son **ficticias**: se generaron con plantillas y listas de equipos, ubicaciones de Cartagena y fallas.

---

## 1. Configuración del entorno

In [1]:
import json
import random
from collections import Counter
from pathlib import Path

import pandas as pd
import spacy
from spacy.training import Example, offsets_to_biluo_tags

DIR_DATA = Path.cwd() / 'data'
random.seed(42)   # barajado reproducible

> 🔷 **CRISP-ML(Q) · Fase 1 — Entendimiento del negocio y de los datos.** Formato de las anotaciones, carga y verificación de la alineación.

---

## 2. Formato de las anotaciones

spaCy necesita las entidades como posiciones de caracteres: `(inicio, fin, ETIQUETA)`, donde `fin` es exclusivo (como en `texto[inicio:fin]`). La función `offsets_to_biluo_tags` convierte esas posiciones en etiquetas por token para inspeccionarlas.

In [2]:
nlp_blank = spacy.blank('es')
text = 'Cambio de reconectador en el circuito 12 por fuga de aceite.'

correctas = [(10, 22, 'EQUIPO'), (29, 40, 'UBICACION'), (45, 59, 'FALLA')]
desalineada = [(10, 21, 'EQUIPO')]   # un carácter menos: termina en medio de la palabra

for inicio, fin, etiqueta in correctas:
    print(f'{etiqueta:9s} {text[inicio:fin]!r}')
doc = nlp_blank.make_doc(text)
print('\nBILUO correctas:  ', offsets_to_biluo_tags(doc, correctas))
print('BILUO desalineada:', offsets_to_biluo_tags(doc, desalineada))

EQUIPO    'reconectador'
UBICACION 'circuito 12'
FALLA     'fuga de aceite'

BILUO correctas:   ['O', 'O', 'U-EQUIPO', 'O', 'O', 'B-UBICACION', 'L-UBICACION', 'O', 'B-FALLA', 'I-FALLA', 'L-FALLA', 'O']
BILUO desalineada: ['O', 'O', '-', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']


/usr/local/lib/python3.13/site-packages/spacy/training/iob_utils.py:149: UserWarning: [W030] Some entities could not be aligned in the text "Cambio de reconectador en el circuito 12 por fuga ..." with entities "[(10, 21, 'EQUIPO')]". Use `spacy.training.offsets_to_biluo_tags(nlp.make_doc(text), entities)` to check the alignment. Misaligned entities ('-') will be ignored during training.
  warnings.warn(


> **Interpretación de la salida:** las etiquetas BILUO siguen el esquema *Begin–Inside–Last–Unit–Outside*. *reconectador* es una entidad de un solo token (`U-EQUIPO`); *circuito 12* ocupa dos (`B-UBICACION`, `L-UBICACION`) y *fuga de aceite* tres (`B-`, `I-`, `L-FALLA`). Con la anotación desalineada aparece un guion `-`: la posición no coincide con el límite de un token y spaCy **ignora esa entidad** al entrenar (solo emite la advertencia `W030`, que se ve en la salida). Por eso se verifica todo el archivo antes de entrenar.

---

## 3. Cargar los datos anotados

In [3]:
# Formato: lista de {"text": str, "entities": [[inicio, fin, etiqueta], ...], "plantilla": int}
with open(DIR_DATA / 'ordenes_trabajo_ner.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

print(f'Órdenes de trabajo: {len(data)}')
print('Entidades por etiqueta:', dict(Counter(l for d in data for _, _, l in d['entities'])))
pd.DataFrame([{'texto': d['text'], 'entidades': [d['text'][s:e] + ' → ' + l for s, e, l in d['entities']]}
              for d in data[:6]])

Órdenes de trabajo: 80
Entidades por etiqueta: {'EQUIPO': 69, 'UBICACION': 74, 'FALLA': 65}


,texto,entidades
0,Inspección del seccionador en circuito 7 sin h...,"[seccionador → EQUIPO, circuito 7 → UBICACION]"
1,Cambio de reconectador en la avenida Pedro de ...,"[reconectador → EQUIPO, avenida Pedro de Hered..."
2,Inspección del poste de concreto en subestació...,"[poste de concreto → EQUIPO, subestación Bocag..."
3,Orden cerrada: interruptor normalizado en sube...,"[interruptor → EQUIPO, subestación Bocagrande ..."
4,Se reporta disparo por sobrecorriente en el pa...,"[disparo por sobrecorriente → FALLA, pararrayo..."
5,Mantenimiento correctivo al reconectador de la...,"[reconectador → EQUIPO, avenida Pedro de Hered..."


### 3.1 Verificar la alineación de todas las anotaciones

In [4]:
problemas = 0
for i, entry in enumerate(data):
    doc = nlp_blank.make_doc(entry['text'])
    tags = offsets_to_biluo_tags(doc, [tuple(e) for e in entry['entities']])
    if '-' in tags:
        problemas += 1
        print(f'⚠️ Ejemplo {i} desalineado: {list(zip([t.text for t in doc], tags))}')
print(f'Ejemplos con entidades desalineadas: {problemas} de {len(data)}')

Ejemplos con entidades desalineadas: 0 de 80


> **Interpretación de la salida:** con cero ejemplos desalineados, todas las entidades se usarán al entrenar. Con anotaciones hechas a mano es común encontrar varias; hay que corregirlas antes de seguir.

> 🔷 **CRISP-ML(Q) · Fase 2 — Ingeniería de datos.** Partición por plantillas y registro de etiquetas.

---

## 4. Partición de entrenamiento y prueba por plantillas

Los datos salen de 10 plantillas de redacción. Si se partieran al azar, la misma plantilla quedaría en entrenamiento y en prueba, y el modelo podría acertar solo por reconocer la frase fija (*“Cambio de … en … por …”*), lo que inflaría el F1. Por eso las órdenes de **2 plantillas** se reservan solo para prueba: miden si el modelo reconoce las entidades en **redacciones que nunca vio**.

In [5]:
PLANTILLAS_PRUEBA = {4, 9}   # "El … presenta … y deja sin servicio …" y "Usuario de … reporta … cerca del …"
train_data = [d for d in data if d['plantilla'] not in PLANTILLAS_PRUEBA]
test_data = [d for d in data if d['plantilla'] in PLANTILLAS_PRUEBA]
print(f'Entrenamiento: {len(train_data)}  |  Prueba: {len(test_data)}')
print('Ejemplos de prueba:')
for d in test_data[:3]:
    print('  ', d['text'])

Entrenamiento: 66  |  Prueba: 14
Ejemplos de prueba:
   Usuario de la subestación Bocagrande reporta lectura errónea cerca del poste de concreto.
   Usuario de la vereda Pasacaballos reporta falla de aislamiento cerca del medidor.
   Usuario de la vereda Pasacaballos reporta lectura errónea cerca del aislador.


---

## 5. Crear el pipeline, registrar etiquetas y entrenar

In [6]:
# Modelo vacío en español: solo tokenizador, sin pesos preentrenados
nlp = spacy.blank('es')
ner = nlp.add_pipe('ner')

# Registra las etiquetas presentes en entrenamiento e inicializa los pesos
for entry in train_data:
    for _, _, label in entry['entities']:
        ner.add_label(label)
optimizer = nlp.initialize()
print(f'Pipeline: {nlp.pipe_names}  |  Etiquetas: {ner.labels}')

Pipeline: ['ner']  |  Etiquetas: ('EQUIPO', 'FALLA', 'UBICACION')


> 🔷 **CRISP-ML(Q) · Fase 3 — Ingeniería del modelo.** Bucle de entrenamiento.

In [7]:
n_epochs = 30
for epoch in range(n_epochs):
    random.shuffle(train_data)
    losses = {}
    for entry in train_data:
        # Ejemplo = documento tokenizado + entidades anotadas
        example = Example.from_dict(nlp.make_doc(entry['text']), {'entities': entry['entities']})
        nlp.update([example], sgd=optimizer, drop=0.2, losses=losses)
    if epoch % 5 == 0 or epoch == n_epochs - 1:
        print(f'Época {epoch:2d}  pérdida NER: {losses["ner"]:.4f}')

Época  0  pérdida NER: 286.2230


Época  5  pérdida NER: 5.7291


Época 10  pérdida NER: 0.1633


Época 15  pérdida NER: 0.0001


Época 20  pérdida NER: 0.0001


Época 25  pérdida NER: 3.4670


Época 29  pérdida NER: 11.3833


> **Interpretación de la salida:** la pérdida cae rápido a casi cero: con 66 oraciones de 8 plantillas el modelo se ajusta por completo al entrenamiento. Los repuntes en las últimas épocas (en la ejecución de referencia, de ~0.0001 a ~11) se deben al *dropout* y al tamaño de paso; no indican que el modelo empeore en general. `drop=0.2` apaga al azar el 20 % de las neuronas en cada paso (*dropout*) para reducir la memorización. Que la pérdida baje solo dice que el modelo se ajusta al entrenamiento; la sección 6 mide si generaliza.

> 🔷 **CRISP-ML(Q) · Fase 4 — Aseguramiento de calidad del modelo.** Precisión, *recall* y F1 por etiqueta en entrenamiento y en prueba, y análisis de errores.

---

## 6. Evaluación

`nlp.evaluate()` compara las entidades predichas con las anotadas y calcula precisión, *recall* y F1 **por entidad completa** (texto y etiqueta deben coincidir).

In [8]:
def evaluar(datos):
    ejemplos = [Example.from_dict(nlp.make_doc(d['text']), {'entities': d['entities']}) for d in datos]
    s = nlp.evaluate(ejemplos)
    fila = {'P': s['ents_p'], 'R': s['ents_r'], 'F1': s['ents_f']}
    por_tipo = {f'F1 {k}': v['f'] for k, v in s['ents_per_type'].items()}
    return {**fila, **por_tipo}

resultados = pd.DataFrame({'entrenamiento': evaluar(train_data), 'prueba (plantillas nuevas)': evaluar(test_data)}).T.round(3)
resultados

,P,R,F1,F1 FALLA,F1 EQUIPO,F1 UBICACION
entrenamiento,1.000,1.000,1.0,1.000,1.000,1.000
prueba (plantillas nuevas),0.603,0.833,0.7,0.619,0.667,0.824


> **Interpretación de la salida:** el F1 de entrenamiento suele ser casi perfecto porque el modelo vio esas oraciones. El de prueba es la cifra honesta: mide si reconoce equipos, ubicaciones y fallas cuando cambia la redacción. Una caída grande indica que el modelo aprendió las plantillas más que las entidades; mira por etiqueta cuál sufre más.

In [9]:
# Errores en prueba: entidades anotadas que faltan y entidades predichas de más
for d in test_data:
    doc = nlp(d['text'])
    gold = {(d['text'][s:e], l) for s, e, l in d['entities']}
    pred = {(ent.text, ent.label_) for ent in doc.ents}
    if gold != pred:
        print(d['text'])
        if gold - pred: print('   faltan :', sorted(gold - pred))
        if pred - gold: print('   de más :', sorted(pred - gold))

Usuario de la subestación Bocagrande reporta lectura errónea cerca del poste de concreto.
   de más : [('cerca', 'FALLA')]
Usuario de la vereda Pasacaballos reporta falla de aislamiento cerca del medidor.
   de más : [('cerca', 'FALLA')]
Usuario de la vereda Pasacaballos reporta lectura errónea cerca del aislador.
   de más : [('cerca', 'FALLA')]
El cable de media tensión presenta lectura errónea y deja sin servicio la zona industrial de Mamonal.
   faltan : [('cable de media tensión', 'EQUIPO')]
   de más : [('El cable', 'UBICACION'), ('deja sin', 'FALLA'), ('media tensión', 'FALLA')]
Usuario del circuito 12 reporta ruido anormal cerca del transformador.
   faltan : [('ruido anormal', 'FALLA')]
   de más : [('ruido anormal cerca', 'FALLA')]
Usuario del barrio Crespo reporta falla de aislamiento cerca del seccionador.
   de más : [('cerca', 'FALLA')]
Usuario de la subestación Bocagrande reporta fuga de aceite cerca del interruptor.
   de más : [('cerca', 'FALLA')]
Usuario del circuito 

> **Interpretación de la salida:** cada bloque es una orden de prueba con errores. En la ejecución de referencia aparecen dos patrones: (1) en *“Usuario de … reporta … **cerca** del …”* el modelo marca *cerca* como `FALLA`, porque en entrenamiento la palabra que sigue a una falla nunca era *cerca*; (2) en *“El cable de media tensión presenta …”* toma *El cable* como `UBICACION`: en las plantillas de entrenamiento ninguna orden **empieza** con el equipo, así que el modelo asocia el inicio de la oración con otra etiqueta. Ambos errores muestran que el modelo aprendió en parte la **posición** de las palabras en las plantillas y no solo las entidades. **Límite importante:** los datos son sintéticos y usan solo 12 valores por etiqueta, así que el modelo nunca ve un equipo nuevo; con órdenes reales habría que anotar cientos de ejemplos con vocabulario variado y evaluar también con equipos no vistos.

> 🔷 **CRISP-ML(Q) · Fase 5 — Despliegue.** Guardado, recarga e inferencia.

---

## 7. Guardar, recargar y usar el modelo

In [10]:
model_path = Path('modelo_ner_ordenes')
nlp.to_disk(model_path)
print(f'Modelo guardado en: {model_path.resolve()}')

nlp2 = spacy.load(model_path)
orden_nueva = 'Se detecta vibración excesiva en el transformador de la subestación Ternera.'
print(f'\nOrden nueva: {orden_nueva}')
for ent in nlp2(orden_nueva).ents:
    print(f'  [{ent.label_}] {ent.text!r}')

# El modelo recargado debe dar los mismos resultados que el original
assert [(e.text, e.label_) for e in nlp2(orden_nueva).ents] == [(e.text, e.label_) for e in nlp(orden_nueva).ents]

Modelo guardado en: /home/claude/upd/run/modelo_ner_ordenes



Orden nueva: Se detecta vibración excesiva en el transformador de la subestación Ternera.
  [FALLA] 'vibración excesiva'
  [EQUIPO] 'transformador'
  [UBICACION] 'subestación Ternera'


> **Interpretación de la salida:** el modelo recargado (`nlp2`) se comporta igual que el original (lo verifica el `assert`) y extrae las entidades de una orden con una redacción nueva. En un sistema real, estas entidades alimentarían las estadísticas de fallas por tipo de equipo.

---

> 🔷 **CRISP-ML(Q) · Fase 6 — Monitoreo y mantenimiento** (fuera de alcance): registrar las entidades detectadas por tipo, revisar periódicamente una muestra de órdenes y vigilar equipos o fallas que el modelo no reconoce. Cada error corregido a mano es una nueva anotación para reentrenar.